In [1]:
import io
import requests
from bs4 import BeautifulSoup
import pandas as pd
import yfinance as yf

# Question 2. [Macro] Indexes YTD (as of 21 August 2026)
# How many indexes (out of 10) have better year-to-date returns than the US (S&P 500) as of August 21, 2026?


stock = dict()

stockList = [
    ["United States - S&P 500", '^GSPC'],
    ["China - Shanghai Composite", '000001.SS'],
    ["Hong Kong - HANG SENG INDEX", '^HSI'],
    ["Australia - S&P/ASX 200", '^AXJO'],
    ["India - Nifty 50", '^NSEI'],
    ["Canada - S&P/TSX Composite", '^GSPTSE'],
    ["Germany - DAX", '^GDAXI'],
    ["United Kingdom - FTSE 100", '^FTSE'],
    ["Japan - Nikkei 225", '^N225'],
    ["Mexico - IPC Mexico", '^MXX'],
    ["Brazil - Ibovespa", '^BVSP'],
]

# Step 1: download daily history for each index and store it
for name, ticker in stockList:
    df = yf.Ticker(ticker).history(start="2026-01-01", end="2026-08-21", interval="1d")
    stock[name] = df

# Step 2: compute YTD return for each index
results = []
for name, ticker in stockList:
    df = stock[name].dropna()          # drop rows with missing close (holidays etc.)
    if df.empty:
        print(f"No data for {name}")
        continue
    start_price = df['Close'].iloc[0]  # first trading day close in the window
    end_price   = df['Close'].iloc[-1] # last trading day close in the window
    ytd_return = (end_price / start_price - 1) * 100
    results.append((name, ytd_return))

# Step 3: put into a DataFrame and compare against the US
# print(results)
returns_df = pd.DataFrame(results, columns=["Index", "YTD Return (%)"])
returns_df = returns_df.sort_values("YTD Return (%)", ascending=False).reset_index(drop=True)

# print(returns_df)
us_return = returns_df.loc[returns_df["Index"] == "United States - S&P 500", "YTD Return (%)"].values[0]

# print(us_return)

better_than_us = returns_df[
    (returns_df["Index"] != "United States - S&P 500") &
    (returns_df["YTD Return (%)"] > us_return)
]

# print(returns_df)
print(f"\nS&P 500 YTD return: {us_return:.2f}%")
print(f"Number of indexes (out of {len(stockList)-1}) beating the US: {len(better_than_us)}")
print(better_than_us)


S&P 500 YTD return: 11.41%
Number of indexes (out of 10) beating the US: 2
                        Index  YTD Return (%)
0          Japan - Nikkei 225       27.750745
1  Canada - S&P/TSX Composite       14.057466
